In [ ]:
from langchain_community.document_loaders import PyPDFLoader

loader = PyPDFLoader('data/sample-return-refund-policy-template.pdf')

documents = loader.load()

# len(documents)


In [49]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=50,
    chunk_overlap=10
)

chunks = text_splitter.split_documents(documents)
# for i, chunk in enumerate(chunks[:5]):
#     print("=" * 80)
#     print(f"Chunk {i+1}")
#     print(chunk.page_content)
len(chunks)

110

In [31]:
from langchain_huggingface import HuggingFaceEmbeddings

embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

vector = embedding_model.embed_query(
    "What is React?"
)

vector[:10]

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7945.66it/s]


[-0.0972486212849617,
 0.007738543674349785,
 0.008727134205400944,
 0.06092529371380806,
 0.09238026291131973,
 0.07291444391012192,
 0.06241093575954437,
 0.07226613163948059,
 0.025867925956845284,
 0.02534273825585842]

In [ ]:
vector_chunks = embedding_model.embed_documents(
    [chunk.page_content for chunk in chunks]
)
# len(vector_chunks)

384

In [83]:
from langchain_core.vectorstores import InMemoryVectorStore

question = "how to contact you?"
vector_store = InMemoryVectorStore(
    embedding=embedding_model
)
vector_store.add_documents(chunks)
retrieved_docs = vector_store.similarity_search(
    question,
    k=3
)


In [84]:
context = "\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

print(context)

Contact Us

on our website:[WEBSITE_CONTACT_PAGE_URL]● By

on our website:[WEBSITE_CONTACT_PAGE_URL]● By


In [91]:
from langchain_core.prompts import ChatPromptTemplate
from langchain.chat_models import init_chat_model

from langchain_core.vectorstores import InMemoryVectorStore

question = "how to contact you?"
vector_store = InMemoryVectorStore(
    embedding=embedding_model
)
vector_store.add_documents(chunks)
retrieved_docs = vector_store.similarity_search(
    question,
    k=3
)
context = "\n\n".join(
    [doc.page_content for doc in retrieved_docs]
)

rag_prompt = ChatPromptTemplate.from_template("""
You are an Enterprise Knowledge Assistant.

Answer ONLY from the provided context.

If the answer is not present in the context,
say:

"I don't have enough information about it."

----------------------------

Context:

{context}

----------------------------

Question:

{question}

Answer:
""")

formatted = rag_prompt.invoke({
    "context":context,
    "question":question    
})

formatted


model = init_chat_model(model="groq:llama-3.3-70b-versatile")

chain = rag_prompt | model
res = chain.invoke({
    "context":context,
    "question":question 
})
res.content

'You can contact us on our website: [WEBSITE_CONTACT_PAGE_URL]'